In [11]:
print('init')

init


# Phase 1 - Data Cleaning and Station Mapping (Train 18238)

This section builds a clean, chronologically ordered dataset for train 18238 and assigns a stable halt-only station index using the static route JSON.

In [17]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

TRAIN_NO = 18238
DATA_PATH = Path("../dataset/supabase/rail_data_rows_27march.csv")
STATIC_ROUTE_PATH = Path("../docs/statics/18238_static.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at: {DATA_PATH.resolve()}")
if not STATIC_ROUTE_PATH.exists():
    raise FileNotFoundError(f"Could not find static route at: {STATIC_ROUTE_PATH.resolve()}")

raw_df = pd.read_csv(DATA_PATH)

# Drop columns not needed for modeling.
cols_to_drop = ["created_at", "updated_at"]
raw_df = raw_df.drop(columns=cols_to_drop, errors="ignore")

# Filter to train 18238 and keep key columns.
train_df = raw_df[raw_df["train_no"] == TRAIN_NO].copy()
if train_df.empty:
    raise ValueError(f"No rows found for train_no={TRAIN_NO}")

# Parse journey_date as date for consistent grouping.
train_df["journey_date"] = pd.to_datetime(train_df["journey_date"], errors="coerce").dt.date
train_df = train_df.dropna(subset=["journey_date", "station_code"])

# Keep latest record per journey and station (polling can create duplicates).
if "id" in train_df.columns:
    train_df = train_df.sort_values(["journey_date", "station_code", "id"])
    train_df = train_df.drop_duplicates(subset=["journey_date", "station_code"], keep="last")

# Ensure delay_departure_minutes is numeric for modeling.
train_df["delay_departure_minutes"] = pd.to_numeric(
    train_df["delay_departure_minutes"], errors="coerce"
 )

print(f"Train rows after cleaning: {len(train_df):,}")
print(f"Journeys available: {train_df['journey_date'].nunique()}")
print(f"Stations observed: {train_df['station_code'].nunique()}")

display(train_df.head(10))

Train rows after cleaning: 5,610
Journeys available: 66
Stations observed: 85


,id,train_no,journey_date,station_code,scheduled_arrival,scheduled_departure,actual_arrival,actual_departure,delay_arrival_minutes,delay_departure_minutes,distance_from_origin_km
394,395,18238,2026-01-17,AGC,1768700700,1.768701e+09,1.768709e+09,1.768710e+09,137.0,148.0,NaN
429,430,18238,2026-01-17,AGN,1768778760,1.768779e+09,1.768781e+09,1.768781e+09,39.0,41.0,NaN
418,419,18238,2026-01-17,AMLA,1768755300,1.768755e+09,1.768757e+09,1.768757e+09,20.0,24.0,NaN
356,357,18238,2026-01-17,ASR,1768645500,1.768646e+09,NaN,1.768646e+09,NaN,16.0,NaN
402,403,18238,2026-01-17,BAB,1768722360,1.768722e+09,1.768725e+09,1.768726e+09,49.0,53.0,NaN
409,410,18238,2026-01-17,BAQ,1768734420,1.768735e+09,1.768738e+09,1.768738e+09,52.0,59.0,NaN
358,359,18238,2026-01-17,BEAS,1768647480,1.768648e+09,1.768649e+09,1.768649e+09,21.0,26.0,NaN
410,411,18238,2026-01-17,BHS,1768735980,1.768736e+09,1.768740e+09,1.768740e+09,63.0,69.0,NaN
407,408,18238,2026-01-17,BINA,1768731900,1.768732e+09,1.768735e+09,1.768735e+09,44.0,52.0,NaN
434,435,18238,2026-01-17,BPHB,1768789260,1.768789e+09,1.768790e+09,1.768790e+09,8.0,11.0,NaN


In [18]:
with open(STATIC_ROUTE_PATH, "r", encoding="utf-8") as f:
    static_json = json.load(f)

route_rows = static_json.get("data", {}).get("route", [])
halt_route = [r for r in route_rows if int(r.get("isHalt", 0)) == 1]
halt_station_order = [
    r.get("stationCode")
    for r in sorted(halt_route, key=lambda x: x.get("sequence", 10**9))
    if r.get("stationCode")
]

if len(halt_station_order) < 2:
    raise ValueError("Static route does not contain enough halt stations.")

station_index_map = {code: idx for idx, code in enumerate(halt_station_order)}

# Keep only halt stations from the static route and assign station_index.
train_df = train_df[train_df["station_code"].isin(station_index_map)].copy()
train_df["station_index"] = train_df["station_code"].map(station_index_map)

train_df = train_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

missing_halts = [s for s in halt_station_order if s not in set(train_df["station_code"]) ]

print(f"Halt stations in static route: {len(halt_station_order)}")
print(f"Halt stations present in data: {train_df['station_code'].nunique()}")
print(f"Missing halt stations in data: {len(missing_halts)}")
if missing_halts:
    print(missing_halts)

print("\nSample ordered rows:")
display(train_df.head(12))

Halt stations in static route: 85
Halt stations present in data: 85
Missing halt stations in data: 0

Sample ordered rows:


,id,train_no,journey_date,station_code,scheduled_arrival,scheduled_departure,actual_arrival,actual_departure,delay_arrival_minutes,delay_departure_minutes,distance_from_origin_km,station_index
0,357,18238,2026-01-17,ASR,1768645500,1.768646e+09,NaN,1.768646e+09,NaN,16.0,NaN,0
1,358,18238,2026-01-17,JNL,1768646340,1.768646e+09,1.768647e+09,1.768648e+09,15.0,19.0,NaN,1
2,359,18238,2026-01-17,BEAS,1768647480,1.768648e+09,1.768649e+09,1.768649e+09,21.0,26.0,NaN,2
3,360,18238,2026-01-17,JUC,1768649880,1.768650e+09,1.768651e+09,1.768652e+09,22.0,26.0,NaN,3
4,361,18238,2026-01-17,JRC,1768651740,1.768652e+09,1.768652e+09,1.768653e+09,11.0,14.0,NaN,4
5,362,18238,2026-01-17,PGW,1768652580,1.768653e+09,1.768653e+09,1.768654e+09,13.0,17.0,NaN,5
6,363,18238,2026-01-17,PHR,1768653780,1.768654e+09,1.768655e+09,1.768655e+09,14.0,17.0,NaN,6
7,364,18238,2026-01-17,LDH,1768655100,1.768656e+09,1.768656e+09,1.768657e+09,12.0,22.0,NaN,7
8,365,18238,2026-01-17,SNL,1768658280,1.768658e+09,1.768658e+09,1.768659e+09,-1.0,3.0,NaN,8
9,366,18238,2026-01-17,DOA,1768658760,1.768659e+09,1.768659e+09,1.768659e+09,4.0,6.0,NaN,9


# Phase 1.4 - Feature Engineering (Train 18238)

This section creates ML-ready features for next-station delay prediction. Features are computed using past-only information within each journey.

In [19]:
# Build base features and target for next-station delay prediction
# Target: delay_departure_minutes at station i+1

feature_df = train_df.copy()

# Ensure chronological ordering within each journey
feature_df = feature_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

# Core lag features within each journey
feature_df["current_delay"] = feature_df["delay_departure_minutes"]
feature_df["previous_delay_1"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(1)
feature_df["previous_delay_2"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(2)

# Rolling features (past-only)
feature_df["rolling_avg_last_3_delays"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)
feature_df["rolling_std_last_3_delays"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=2).std())
)

# Trend slope over last 3 delays (simple linear fit)
def _trend_slope(values: pd.Series) -> float:
    if values.isna().all():
        return np.nan
    y = values.values.astype(float)
    x = np.arange(len(y), dtype=float)
    if len(y) < 2:
        return np.nan
    # simple slope of least squares line
    x_mean = x.mean()
    y_mean = np.nanmean(y)
    denom = ((x - x_mean) ** 2).sum()
    if denom == 0:
        return np.nan
    return float(np.nansum((x - x_mean) * (y - y_mean)) / denom)

feature_df["delay_trend_slope"] = (
    feature_df.groupby("journey_date")["delay_departure_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=2).apply(_trend_slope, raw=False))
)

# Station progression features
max_station_index = feature_df["station_index"].max()
feature_df["journey_progress_pct"] = feature_df["station_index"] / max_station_index

# Temporal features from journey_date (proxy; for richer time features use actual timestamps)
feature_df["journey_date_dt"] = pd.to_datetime(feature_df["journey_date"], errors="coerce")
feature_df["day_of_week"] = feature_df["journey_date_dt"].dt.dayofweek
feature_df["month"] = feature_df["journey_date_dt"].dt.month
feature_df["is_weekend"] = feature_df["day_of_week"].isin([5, 6]).astype(int)

# Target: next station delay
feature_df["target_next_delay"] = feature_df.groupby("journey_date")["delay_departure_minutes"].shift(-1)
feature_df["next_station_index"] = feature_df.groupby("journey_date")["station_index"].shift(-1)

# Remove rows where we cannot form a next-station target
feature_df = feature_df.dropna(subset=["target_next_delay", "next_station_index"]).copy()
feature_df["next_station_index"] = feature_df["next_station_index"].astype(int)

# Basic sanity checks
print(f"Feature rows: {len(feature_df):,}")
print(f"Journeys covered: {feature_df['journey_date'].nunique()}")
print("Feature null rates (top 10):")
null_rates = feature_df.isna().mean().sort_values(ascending=False)
print(null_rates.head(10))

# Preview selected features
preview_cols = [
    "journey_date",
    "station_code",
    "station_index",
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "target_next_delay",
]

display(feature_df[preview_cols].head(12))

Feature rows: 5,153
Journeys covered: 66
Feature null rates (top 10):
distance_from_origin_km      1.000000
previous_delay_1             0.075490
previous_delay_2             0.075296
delay_departure_minutes      0.062488
current_delay                0.062488
actual_departure             0.062488
rolling_std_last_3_delays    0.026392
delay_trend_slope            0.026392
delay_arrival_minutes        0.020959
actual_arrival               0.020959
dtype: float64


,journey_date,station_code,station_index,current_delay,previous_delay_1,previous_delay_2,rolling_avg_last_3_delays,rolling_std_last_3_delays,delay_trend_slope,journey_progress_pct,day_of_week,month,is_weekend,target_next_delay
0,2026-01-17,ASR,0,16.0,NaN,NaN,NaN,NaN,NaN,0.000000,5,1,1,19.0
1,2026-01-17,JNL,1,19.0,16.0,NaN,16.000000,NaN,NaN,0.011905,5,1,1,26.0
2,2026-01-17,BEAS,2,26.0,19.0,16.0,17.500000,2.121320,0.75,0.023810,5,1,1,26.0
3,2026-01-17,JUC,3,26.0,26.0,19.0,20.333333,5.131601,5.00,0.035714,5,1,1,14.0
4,2026-01-17,JRC,4,14.0,26.0,26.0,23.666667,4.041452,3.50,0.047619,5,1,1,17.0
5,2026-01-17,PGW,5,17.0,14.0,26.0,22.000000,6.928203,-6.00,0.059524,5,1,1,17.0
6,2026-01-17,PHR,6,17.0,17.0,14.0,19.000000,6.244998,-4.50,0.071429,5,1,1,22.0
7,2026-01-17,LDH,7,22.0,17.0,17.0,16.000000,1.732051,1.50,0.083333,5,1,1,3.0
8,2026-01-17,SNL,8,3.0,22.0,17.0,18.666667,2.886751,2.50,0.095238,5,1,1,6.0
9,2026-01-17,DOA,9,6.0,3.0,22.0,14.000000,9.848858,-7.00,0.107143,5,1,1,9.0


In [26]:
# Additional required features (distance, temporal, segment, historical stats)
feature_df = feature_df.sort_values(["journey_date", "station_index"]).reset_index(drop=True)

# Static route metadata (halt-only) for distance/schedule features
halt_meta = {r.get("stationCode"): r for r in halt_route if r.get("stationCode")}

def _abs_minutes(meta: dict, key: str) -> float:
    if not meta:
        return np.nan
    val = meta.get(key)
    if val is None:
        return np.nan
    day = meta.get("day", 1)
    return (day - 1) * 1440 + float(val)

# Distance features
if "distance_from_origin_km" in feature_df.columns:
    feature_df["distance_from_origin"] = pd.to_numeric(
        feature_df["distance_from_origin_km"], errors="coerce"
    )
else:
    feature_df["distance_from_origin"] = np.nan

# If dataset distance is missing, use static route distance from source
if feature_df["distance_from_origin"].isna().all():
    feature_df["distance_from_origin"] = feature_df["station_code"].map(
        lambda c: halt_meta.get(c, {}).get("distanceFromSourceKm")
    )

if feature_df["distance_from_origin"].notna().any():
    max_dist = feature_df["distance_from_origin"].max()
    if pd.notna(max_dist) and max_dist != 0:
        feature_df["journey_progress_pct"] = feature_df["distance_from_origin"] / max_dist

# Static schedule features (minutes from journey start)
feature_df["scheduled_arrival_min"] = feature_df["station_code"].map(
    lambda c: _abs_minutes(halt_meta.get(c), "scheduledArrival")
 )
feature_df["scheduled_departure_min"] = feature_df["station_code"].map(
    lambda c: _abs_minutes(halt_meta.get(c), "scheduledDeparture")
 )
feature_df["halt_duration_min"] = feature_df["station_code"].map(
    lambda c: halt_meta.get(c, {}).get("haltDurationMinutes")
 )
feature_df["speed_on_section_kmph"] = feature_df["station_code"].map(
    lambda c: halt_meta.get(c, {}).get("speedOnSectionKmph")
 )

# Temporal features (hour_of_day, overnight_segment_flag)
time_candidates = [
    "actual_departure",
    "actual_arrival",
    "scheduled_departure",
    "scheduled_arrival",
    "expected_departure",
    "expected_arrival",
    "departure_time",
    "arrival_time",
    "departure",
    "arrival",
]
time_col = next((c for c in time_candidates if c in feature_df.columns), None)
if time_col:
    time_parsed = pd.to_datetime(feature_df[time_col], errors="coerce")
    if time_parsed.isna().all():
        time_parsed = pd.to_datetime(
            feature_df["journey_date"].astype(str)
            + " "
            + feature_df[time_col].astype(str),
            errors="coerce",
        )
    feature_df["hour_of_day"] = time_parsed.dt.hour
else:
    feature_df["hour_of_day"] = np.nan

feature_df["overnight_segment_flag"] = feature_df["hour_of_day"].isin(
    [22, 23, 0, 1, 2, 3, 4]
).astype(float)

# Segment features
feature_df["distance_to_next_station"] = (
    feature_df.groupby("journey_date")["distance_from_origin"].shift(-1)
    - feature_df["distance_from_origin"]
)

# Compute scheduled travel time using static schedule when available
next_arrival_min = feature_df.groupby("journey_date")["scheduled_arrival_min"].shift(-1)
feature_df["scheduled_travel_time_min"] = next_arrival_min - feature_df["scheduled_departure_min"]

feature_df["average_speed_expected_kmph"] = np.where(
    feature_df["scheduled_travel_time_min"].gt(0),
    feature_df["distance_to_next_station"]
    / (feature_df["scheduled_travel_time_min"] / 60.0),
    np.nan,
)
feature_df["average_speed_expected_kmph"] = feature_df[
    "average_speed_expected_kmph"
].replace([np.inf, -np.inf], np.nan)

# If average speed is missing, fall back to static speed on section
feature_df["average_speed_expected_kmph"] = feature_df[
    "average_speed_expected_kmph"
].fillna(feature_df["speed_on_section_kmph"])

# Historical statistical features (past-only expanding windows)
feature_df["next_station_code"] = feature_df.groupby("journey_date")["station_code"].shift(-1)
feature_df["segment_key"] = (
    feature_df["station_code"].astype(str)
    + "->"
    + feature_df["next_station_code"].astype(str)
 )

feature_df["avg_delay_at_station"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().mean())
feature_df["delay_variance_at_station"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().var())
feature_df["avg_delay_for_train"] = feature_df["delay_departure_minutes"].shift(1).expanding().mean()
feature_df["station_congestion_score"] = feature_df.groupby("station_code")[
    "delay_departure_minutes"
].transform(lambda s: s.shift(1).expanding().mean().abs())
feature_df["avg_delay_on_segment"] = feature_df.groupby("segment_key")[
    "target_next_delay"
].transform(lambda s: s.shift(1).expanding().mean())

# Feature Catalog (Train 18238)

| Feature | Description | Calculation / Source |
| --- | --- | --- |
| `current_delay` | Delay at the current station. | `delay_departure_minutes` at station $i$. |
| `previous_delay_1` | Delay at the previous station. | Shifted by 1 within each `journey_date`. |
| `previous_delay_2` | Delay two stations back. | Shifted by 2 within each `journey_date`. |
| `rolling_avg_last_3_delays` | Recent average delay trend. | Mean of the last 3 delays (past-only), per journey. |
| `rolling_std_last_3_delays` | Recent delay volatility. | Std dev of the last 3 delays (past-only), per journey. |
| `delay_trend_slope` | Recent trend direction. | Least-squares slope on last 3 past delays. |
| `station_index` | Station order along route. | Halt-only index from static route order. |
| `journey_progress_pct` | Progress along route. | Distance-from-source / max distance (or station index ratio if distance missing). |
| `day_of_week` | Weekday signal. | `journey_date.dt.dayofweek` (0=Mon). |
| `month` | Seasonality signal. | `journey_date.dt.month`. |
| `is_weekend` | Weekend flag. | 1 if day is Sat/Sun else 0. |
| `hour_of_day` | Time-of-day signal. | From actual/scheduled time column when available. |
| `overnight_segment_flag` | Late-night segment. | 1 if hour in {22,23,0,1,2,3,4}. |
| `distance_from_origin` | Distance from route start. | Static `distanceFromSourceKm` fallback if dataset distance is missing. |
| `distance_to_next_station` | Segment length. | Next station distance minus current distance within journey. |
| `scheduled_travel_time_min` | Scheduled segment travel time. | Next station scheduled arrival - current scheduled departure (static). |
| `halt_duration_min` | Scheduled halt at station. | Static `haltDurationMinutes`. |
| `average_speed_expected_kmph` | Expected segment speed. | Distance to next / scheduled travel time; fallback to static `speedOnSectionKmph`. |
| `avg_delay_at_station` | Typical delay at station. | Expanding mean of past delays per station (shifted). |
| `delay_variance_at_station` | Delay variability at station. | Expanding variance of past delays per station (shifted). |
| `avg_delay_for_train` | Overall train delay tendency. | Expanding mean of past delays across all rows (shifted). |
| `station_congestion_score` | Station delay pressure proxy. | Expanding mean of abs past delays per station (shifted). |
| `avg_delay_on_segment` | Typical delay for segment. | Expanding mean of past `target_next_delay` per segment (shifted). |

In [27]:
# Validation checks for feature engineering
expected_cols = [
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_of_day",
    "overnight_segment_flag",
    "distance_from_origin",
    "distance_to_next_station",
    "scheduled_travel_time_min",
    "halt_duration_min",
    "average_speed_expected_kmph",
    "avg_delay_at_station",
    "avg_delay_on_segment",
    "avg_delay_for_train",
    "station_congestion_score",
    "delay_variance_at_station",
    "target_next_delay",
    "station_index",
    "next_station_index",
]

missing_cols = [c for c in expected_cols if c not in feature_df.columns]
print(f"Missing feature columns: {missing_cols}")

null_summary = feature_df[expected_cols].isna().mean().sort_values(ascending=False).head(12)
print("Top null rates:")
print(null_summary)

step = feature_df["next_station_index"] - feature_df["station_index"]
print("Next-station step distribution (top 5):")
print(step.value_counts().head(5))
print(f"Non-positive steps: {(step <= 0).sum()}")

# Spot-check first station rows per journey for expected lag nulls
first_station_rows = feature_df.groupby("journey_date").head(1)
print("First-station lag null rate:")
print(first_station_rows[["previous_delay_1", "previous_delay_2"]].isna().mean())

Missing feature columns: []
Top null rates:
delay_variance_at_station    0.083447
previous_delay_1             0.075490
previous_delay_2             0.075296
avg_delay_at_station         0.067533
station_congestion_score     0.067533
hour_of_day                  0.062488
current_delay                0.062488
avg_delay_on_segment         0.029109
rolling_std_last_3_delays    0.026392
delay_trend_slope            0.026392
distance_to_next_station     0.025616
journey_progress_pct         0.012808
dtype: float64
Next-station step distribution (top 5):
1    5153
Name: count, dtype: int64
Non-positive steps: 0
First-station lag null rate:
previous_delay_1    1.0
previous_delay_2    1.0
dtype: float64


# Phase 2 - Baselines (Train 18238)

This section builds simple baselines using a time-based split on journey dates to avoid leakage.

In [28]:
# Time-based split by journey_date
unique_dates = sorted(feature_df["journey_date"].unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough journeys for a time-based split.")

split_idx = int(len(unique_dates) * 0.8)
train_dates = set(unique_dates[:split_idx])
test_dates = set(unique_dates[split_idx:])

train_set = feature_df[feature_df["journey_date"].isin(train_dates)].copy()
test_set = feature_df[feature_df["journey_date"].isin(test_dates)].copy()

print(f"Train journeys: {len(train_dates)} | Test journeys: {len(test_dates)}")
print(f"Train rows: {len(train_set):,} | Test rows: {len(test_set):,}")

feature_cols = [
    "current_delay",
    "previous_delay_1",
    "previous_delay_2",
    "rolling_avg_last_3_delays",
    "rolling_std_last_3_delays",
    "delay_trend_slope",
    "journey_progress_pct",
    "day_of_week",
    "month",
    "is_weekend",
    "hour_of_day",
    "overnight_segment_flag",
    "station_index",
    "distance_from_origin",
    "distance_to_next_station",
    "scheduled_travel_time_min",
    "halt_duration_min",
    "average_speed_expected_kmph",
    "avg_delay_at_station",
    "avg_delay_on_segment",
    "avg_delay_for_train",
    "station_congestion_score",
    "delay_variance_at_station",
]

usable_feature_cols = [c for c in feature_cols if train_set[c].notna().any()]
print(f"Features used: {len(usable_feature_cols)}")

# Prepare model sets and impute missing feature values with train medians
train_model = train_set.dropna(subset=["target_next_delay"]).copy()
test_model = test_set.dropna(subset=["target_next_delay"]).copy()

median_values = train_model[usable_feature_cols].median(numeric_only=True).fillna(0)
train_model[usable_feature_cols] = train_model[usable_feature_cols].fillna(median_values)
test_model[usable_feature_cols] = test_model[usable_feature_cols].fillna(median_values)

print(f"Model-ready train rows: {len(train_model):,}")
print(f"Model-ready test rows: {len(test_model):,}")

# Baseline predictors
baseline_predictions = {}

# 1) Persistence baseline (predict current delay for next station)
baseline_predictions["persistence"] = test_model["current_delay"].values

# 2) Rolling mean baseline
baseline_predictions["rolling_mean_3"] = test_model["rolling_avg_last_3_delays"].values

# 3) Simple linear regression (numpy least squares with intercept)
X_train = train_model[usable_feature_cols].values
X_test = test_model[usable_feature_cols].values

y_train = train_model["target_next_delay"].values
y_test = test_model["target_next_delay"].values

# Add intercept column
X_train_i = np.column_stack([np.ones(len(X_train)), X_train])
X_test_i = np.column_stack([np.ones(len(X_test)), X_test])

beta, *_ = np.linalg.lstsq(X_train_i, y_train, rcond=None)
lin_pred = X_test_i @ beta
baseline_predictions["linear_lstsq"] = lin_pred

# 4) Ridge regression (closed-form)
alpha = 1.0
XtX = X_train_i.T @ X_train_i
ridge_beta = np.linalg.solve(XtX + alpha * np.eye(XtX.shape[0]), X_train_i.T @ y_train)
ridge_pred = X_test_i @ ridge_beta
baseline_predictions["ridge_alpha_1"] = ridge_pred

# Evaluation metrics

def evaluate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {"mae": mae, "rmse": rmse}

results = []
for name, preds in baseline_predictions.items():
    metrics = evaluate_metrics(y_test, preds)
    results.append({"model": name, **metrics})

results_df = pd.DataFrame(results).sort_values("mae")
results_df.reset_index(drop=True, inplace=True)

print("Baseline results (sorted by MAE):")
display(results_df)

Train journeys: 52 | Test journeys: 14
Train rows: 4,062 | Test rows: 1,091
Features used: 23
Model-ready train rows: 4,062
Model-ready test rows: 1,091
Baseline results (sorted by MAE):


,model,mae,rmse
0,ridge_alpha_1,7.262982,13.075715
1,linear_lstsq,7.265582,13.077093
2,persistence,9.744271,19.043994
3,rolling_mean_3,12.878705,19.567123
